In [1]:
import json
import networkx as nx
from itertools import combinations
from collections import defaultdict
from pathlib import Path
from pyvis.network import Network
import os, sys



from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community import greedy_modularity_communities


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.makedirs(PROJECT_ROOT / "graph_artifacts" / "visuals", exist_ok=True)
sys.path.append(str(PROJECT_ROOT))

from util.utils import load, dump

PROCESSED_VIDEO_DATA_PATH = PROJECT_ROOT / "data" / "video_data_processed.json"
OUTVISUAL = PROJECT_ROOT / "data" / "GraphVisual.html" 
OUTPUTGRAPH = PROJECT_ROOT / "data" / "graph.graphml"

ARTIFACT_DIR = PROJECT_ROOT / "graph_artifacts"
VISUAL_DIR  = ARTIFACT_DIR / "visuals"

In [12]:
# Graph Helper Class
class graph:
# INIT ===========================================================================================================

    def __init__(self, Gr = None, gPath = None):
        self.G = nx.MultiDiGraph()
        self.G = Gr
        if gPath:
            try:
                self.G = nx.read_graphml(path=gPath)
            except FileNotFoundError as e:
                print(f"Tried reading graph from memory, failed. Graph doesn't exist yet.")

# VISUALIZE ======================================================================================================================

    def visualize(self, outPath=OUTVISUAL):
        # Get top nodes
        top_nodes = sorted(self.G.degree(), key=lambda x: x[1], reverse=True)[:1000]
        G_sub = self.G.subgraph([n for n, d in top_nodes])

        net = Network(height="750px", width="100%", directed=False, notebook=False)
        net.from_nx(G_sub)

        # Colour by node type
        for node in net.nodes:
            if self.G.nodes[node["id"]].get("node_type") == "celebrity":
                node["color"] = "#e74c3c"   # celebs
            elif self.G.nodes[node["id"]].get("node_type") == "brand":
                node["color"] = "#3498db"   # brands
            else:
                node["color"] = "#95a5a6"   # else
                
        for edge in net.edges:
            edge["label"] = ""
            edge["title"] = ""
            edge["width"] = 0.5
            edge["color"] = "#cccccc"

        net.set_options("""
        {
        "edges": {
            "arrows": { "to": { "enabled": false } },
            "color": { "color": "#cccccc", "opacity": 0.75 },
            "width": 0.5,
            "smooth": { "enabled": false }
        },
        "physics": {
            "forceAtlas2Based": {
            "gravitationalConstant": -50,
            "springLength": 100
            },
            "solver": "forceAtlas2Based",
            "stabilization": {
                "enabled": true,
                "iterations": 200,
                "fit": true
                }
            }
        }
        """)

        net.show(str(outPath), notebook=False)
        print(f"Graph saved to: {outPath}")

# RUN COMMUNITY DETECTION ======================================================================================================== 

    def communities(self, method="louvain", resolution=1.0, min_weight = 20):
        G_undirected = self.G.to_undirected() if self.G.is_directed() else self.G


        # Filter Edge weights
        if min_weight > 1:
                edges_to_remove = [(u, v) for u, v, d in G_undirected.edges(data=True)
                                if d.get("width", 0) < min_weight]
                G_undirected = G_undirected.copy()
                G_undirected.remove_edges_from(edges_to_remove)
                G_undirected.remove_nodes_from(list(nx.isolates(G_undirected)))
                print(f"After weight filter — Nodes: {G_undirected.number_of_nodes()}, Edges: {G_undirected.number_of_edges()}")

        if G_undirected.number_of_nodes() == 0:
            print(f"Graph is empty after min_weight={min_weight} filter — try a lower threshold.")
            return []


        if method == "louvain":
            result = louvain_communities(G_undirected, seed=42, resolution=resolution)

        elif method == "greedy":
            result = greedy_modularity_communities(G_undirected)

        else:
            #Default to louvian
            result = louvain_communities(G_undirected, seed=42, resolution=resolution)



        # Tag every node with its community ID
        for i, community in enumerate(result):
            for node in community:
                self.G.nodes[node]["community"] = i


        self.communities_result = result

        print(f"Communities found:  {len(result)}")
        print(f"Largest community:  {max(len(c) for c in result)} nodes")
        print(f"Smallest community: {min(len(c) for c in result)} nodes")
        print()
        
        for i, community in enumerate(result):
            celebs = [n for n in community if self.G.nodes[n].get("node_type") == "celebrity"]
            brands = [n for n in community if self.G.nodes[n].get("node_type") == "brand"]
            print(f"  Community {i}: {len(community)} nodes | {len(celebs)} celebs | {len(brands)} brands")
            print(f"    Top members: {sorted(community, key=lambda n: self.G.nodes[n].get('mention_count', 0), reverse=True)[:5]}")

        return result


# EXPORT =========================================================================================================================

    def export(self, outPath = OUTPUTGRAPH):
        nx.write_graphml(self.G, outPath)

In [3]:
with open(PROCESSED_VIDEO_DATA_PATH, "r", encoding="utf-8") as f:
    processed = json.load(f)

comments = processed["comments"]
entity_counts = processed["entity_counts"]

In [4]:
co_mentions = defaultdict(int)
entity_sentiment = defaultdict(list)

# GET CELEBRITY CO-MENTIONS 
for comment in comments:
    entities = list(set(comment.get("celebs", [])))

    if len(entities) < 2:
        continue

    for pair in combinations(sorted(entities), 2):
        co_mentions[pair] += 1


In [5]:
G1 = nx.Graph()

# Add nodes
for celeb, count in entity_counts["celebs"].items():
    G1.add_node(celeb, node_type="celebrity", mention_count=count)

# Add edges
MIN_CO_MENTIONS = 2
for (e1, e2), weight in co_mentions.items():
    if weight >= MIN_CO_MENTIONS:
        G1.add_edge(e1, e2, weight=weight)

# No co-mentiones with anyone
G1.remove_nodes_from(list(nx.isolates(G1)))

print(f"Nodes:          {G1.number_of_nodes()}")
print(f"Edges:          {G1.number_of_edges()}")
print(f"Celebrities:    {sum(1 for _, d in G1.nodes(data=True) if d.get('node_type') == 'celebrity')}")


Nodes:          142
Edges:          3361
Celebrities:    142


In [6]:
co_mentions = defaultdict(int)

for comment in comments:
    celebs = list(set(comment.get("celebs", [])))
    brands = list(set(comment.get("brands", [])))

    for celeb in celebs:
        for brand in brands:
            co_mentions[(celeb, brand)] += 1

G2 = nx.Graph()

for celeb, count in entity_counts["celebs"].items():
    G2.add_node(celeb, node_type="celebrity", mention_count=count)

for brand, count in entity_counts["brands"].items():
    G2.add_node(brand, node_type="brand", mention_count=count)

MIN_CO_MENTIONS = 2
for (celeb, brand), weight in co_mentions.items():
    if weight >= MIN_CO_MENTIONS:
        G2.add_edge(celeb, brand, weight=weight)

G2.remove_nodes_from(list(nx.isolates(G2)))

print(f"Nodes:       {G2.number_of_nodes()}")
print(f"Edges:       {G2.number_of_edges()}")
print(f"Celebrities: {sum(1 for _, d in G2.nodes(data=True) if d.get('node_type') == 'celebrity')}")
print(f"Brands:      {sum(1 for _, d in G2.nodes(data=True) if d.get('node_type') == 'brand')}")

Nodes:       83
Edges:       120
Celebrities: 51
Brands:      32


In [7]:
CG = graph(G1)

# EXPORT CELEB GRAPH
celebVisual = VISUAL_DIR / "celebs.html"
celebGraph = ARTIFACT_DIR / "celebs.graphml"
CG.visualize(outPath=celebVisual)
CG.export(outPath=celebGraph)

c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\celebs.html
Graph saved to: c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\celebs.html


In [8]:
CBG = graph(G2)

# EXPORT CELEB - BRAND GRAPH 
celebBrandVisual = VISUAL_DIR / "CBG.html"
celebBrandGraph = ARTIFACT_DIR / "CBG.graphml"
CBG.visualize(outPath=celebBrandVisual)
CBG.export(outPath=celebBrandGraph)



c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\CBG.html
Graph saved to: c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\CBG.html


In [13]:
celebCommunities = CG.communities(method="greedy", min_weight=20)
communities_serializable = [list(community) for community in celebCommunities]
dump(communities_serializable, outPath=ARTIFACT_DIR / "celebCommunities.json")

print(f"celebCommnities: {celebCommunities[:10]}")

TypeError: graph.communities() got an unexpected keyword argument 'min_weight'

[('Beyonce', 'Bad Bunny', {'width': 13}), ('Beyonce', 'Blue Ivy', {'width': 32}), ('Beyonce', 'Charli xcx', {'width': 6}), ('Beyonce', 'Ciara', {'width': 23}), ('Beyonce', 'Doechii', {'width': 8})]
